# 03 — Vector Search
## Databricks Expert Agent Project

**What this notebook does:**
1. Creates a Mosaic AI Vector Search endpoint (compute for similarity search)
2. Creates a Delta Sync index over `chatbot.rag_chatbot.doc_chunks`

**Output:** A live Vector Search index that automatically stays in sync with the doc_chunks Delta table. The agent uses this to retrieve the most relevant chunks for any question.

## 1. Configure Vector Search Assets

This cell defines the source chunk table, the Vector Search endpoint, and the Delta Sync index name.

The index uses precomputed embeddings from Notebook 2, so Vector Search does not need to call the embedding model during indexing. It only needs the embedding column, primary key, and source Delta table.

In [0]:
# Unity Catalog location of the chunk table produced by Notebook 2.
CATALOG       = "chatbot"
SCHEMA        = "rag_chatbot"
CHUNKS_TABLE  = f"{CATALOG}.{SCHEMA}.doc_chunks"

# Vector Search assets. The endpoint is shared compute; the index is the
# Databricks-specific retrieval surface used by the app.
VS_ENDPOINT   = "databricks-expert-endpoint"
VS_INDEX      = f"{CATALOG}.{SCHEMA}.doc_chunks_index"

# The index uses embeddings already written into CHUNKS_TABLE.
EMBED_MODEL   = "databricks-gte-large-en"
EMBED_COLUMN  = "embedding"
ID_COLUMN     = "chunk_id"

print(f"Endpoint : {VS_ENDPOINT}")
print(f"Index    : {VS_INDEX}")
print(f"Source   : {CHUNKS_TABLE}")

## 2. Create Or Reuse The Vector Search Endpoint

The endpoint is the serving compute for similarity search. Multiple indexes can share the same endpoint, including the Databricks, Fabric, and Power BI indexes.

This cell is safe to rerun: if the endpoint already exists, the notebook reuses it and waits until it is online.

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)

try:
    vsc.create_endpoint(
        name=VS_ENDPOINT,
        endpoint_type="STANDARD"
    )
    print(f"✓ Endpoint '{VS_ENDPOINT}' created — waiting for it to be ready...")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"✓ Endpoint '{VS_ENDPOINT}' already exists — skipping creation")
    else:
        raise e

# Wait until endpoint is ONLINE before proceeding
import time
while True:
    status = vsc.get_endpoint(VS_ENDPOINT)["endpoint_status"]["state"]
    print(f"  Endpoint state: {status}")
    if status == "ONLINE":
        print("✓ Endpoint is ONLINE")
        break
    time.sleep(20)

## 3. Enable Delta Change Data Feed

Delta Sync indexes require Change Data Feed on the source table so Vector Search can identify inserts, updates, and deletes.

This cell enables CDF on `chatbot.rag_chatbot.doc_chunks` and immediately displays the table property so you can verify it is active.

In [0]:
spark.sql(f"ALTER TABLE {CHUNKS_TABLE} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
print(f"✓ Change Data Feed enabled on {CHUNKS_TABLE}")

# Verify it's on
props = spark.sql(f"SHOW TBLPROPERTIES {CHUNKS_TABLE}").filter("key = 'delta.enableChangeDataFeed'")
display(props)

## 4. Confirm Unity Catalog Visibility

This small diagnostic confirms the notebook can see the expected Unity Catalog catalog.

If this fails, troubleshoot catalog permissions or metastore assignment before trying to create the Vector Search index.

In [0]:
# Run this to see your metastore details
display(spark.sql("SELECT * FROM system.information_schema.catalogs WHERE catalog_name = 'chatbot'"))

## 5. Create Or Reuse The Delta Sync Index

This cell creates the Databricks corpus Vector Search index over the chunk table.

Important settings:

- `pipeline_type="TRIGGERED"` means syncs are explicit or job-driven.
- `primary_key=chunk_id` gives Vector Search a stable row identity.
- `embedding_vector_column=embedding` tells Vector Search to use the vectors generated in Notebook 2.
- `embedding_dimension=1024` must match the embedding model output.

In [0]:
try:
    # Create a Delta Sync index over the chunk table. Because embeddings are
    # precomputed, Vector Search reads the embedding column instead of calling
    # an embedding endpoint during indexing.
    idx = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=VS_INDEX,
        source_table_name=CHUNKS_TABLE,
        pipeline_type="TRIGGERED",
        primary_key=ID_COLUMN,
        embedding_dimension=1024,
        embedding_vector_column=EMBED_COLUMN
    )
    print(f"✓ Index '{VS_INDEX}' created — waiting for initial sync...")
except Exception as e:
    # Reusing the index keeps the notebook idempotent for repeat runs.
    if "already exists" in str(e).lower():
        print(f"✓ Index '{VS_INDEX}' already exists — skipping creation")
    else:
        raise e

# Wait for the first sync to complete so downstream app queries do not hit a
# partially built index.
while True:
    status = vsc.get_index(VS_ENDPOINT, VS_INDEX).describe()
    state = status.get("status", {}).get("ready", False)
    detail = status.get("status", {}).get("message", "syncing...")
    print(f"  Index state: {detail}")
    if state:
        print("✓ Index is READY")
        break
    time.sleep(30)

## 6. Describe The Index

This cell prints the raw index description returned by the Vector Search API.

Use it for troubleshooting pipeline state, endpoint wiring, indexed row counts, and any detailed status messages that are not visible in the short progress loop.

In [0]:
import json

desc = vsc.get_index(VS_ENDPOINT, VS_INDEX).describe()
print(json.dumps(desc, indent=2))

## 7. Poll Sync Progress

This cell gives a compact view of Delta Sync progress.

It is useful after creating the index or triggering a sync because it reports detailed state, indexed rows, sync completion percentage, and synced row counts.

In [0]:
import time, json

for _ in range(20):
    desc = vsc.get_index(VS_ENDPOINT, VS_INDEX).describe()
    status = desc["status"]

    progress = status.get("triggered_update_status", {}).get("triggered_update_progress", {})

    print("State:", status.get("detailed_state"))
    print("Indexed rows:", status.get("indexed_row_count"))
    print("Progress:", progress.get("sync_progress_completion"))
    print("Synced:", progress.get("num_synced_rows"), "/", progress.get("total_rows_to_sync"))
    print("-" * 80)

    if not progress:
        print("No active sync progress found.")
        break

    if progress.get("sync_progress_completion", 0) >= 0.999:
        print("Sync looks complete.")
        break

    time.sleep(30)

## 8. Validate Indexed Source Distribution

This cell checks the chunk table by source and source type after indexing.

It does not query Vector Search directly, but it confirms the table behind the index still contains the expected official-docs and internal-content distribution.

In [0]:
display(spark.sql("""
SELECT source, source_type, COUNT(*) AS chunks
FROM chatbot.rag_chatbot.doc_chunks
GROUP BY source, source_type
ORDER BY chunks DESC
"""))

## 9. Optional Manual Sync Trigger

This skipped cell manually triggers a Delta Sync refresh for the Databricks index.

Use it when the chunk table has changed and you want to refresh the index immediately outside of the normal Lakeflow or job schedule.

In [0]:
%skip
idx = vsc.get_index(VS_ENDPOINT, VS_INDEX)

print("Triggering Vector Search index sync...")
idx.sync()

print("✓ Sync triggered")